# Process Midi
File to process midi files to remove corrupt files and extract features we're interested in. 

In [1]:
# Import packages
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd 

import pretty_midi
import librosa
import mir_eval
import mir_eval.display
import tables
import IPython.display
import os
import json

In [2]:
# Loop through files in raw_data folder to extract data from midi files and store in dataframe
folder = '../data/raw_data/lmd_full'

records = []
errors = []

for root, dirs, files in os.walk(folder):
    for file in files:
        if not file.endswith('.mid'):
            continue
        
        filepath = os.path.join(root, file)
        
        try:
            midi = pretty_midi.PrettyMIDI(filepath)
            
            tempo_times, tempos = midi.get_tempo_changes()
            key_changes = midi.key_signature_changes
            time_sig_changes = midi.time_signature_changes

            records.append({
                'file':               file,
                'filepath':           filepath,
                'duration_s':         round(midi.get_end_time(), 2),
                'resolution':         midi.resolution,
                'n_instruments':      len(midi.instruments),
                'n_notes':            sum(len(i.notes) for i in midi.instruments),
                'n_tempo_changes':    len(tempos),
                'initial_bpm':        round(tempos[0], 2) if len(tempos) > 0 else None,
                'n_key_changes':      len(key_changes),
                'initial_key':        key_changes[0].key_number if len(key_changes) > 0 else None,
                'n_time_sig_changes': len(time_sig_changes),
                'initial_time_sig':   f"{time_sig_changes[0].numerator}/{time_sig_changes[0].denominator}" if len(time_sig_changes) > 0 else None,
            })

        except Exception as e:
            errors.append({'file': filepath, 'error': str(e)})

    # Progress update per subfolder
    if records:
        print(f"Processed {len(records)} files so far (latest folder: {root})...")

df = pd.DataFrame(records)
print(f"\nDone. Successfully parsed: {len(df)} files")
print(f"Errors: {len(errors)} files")

/home/arige/miniconda3/envs/dsan2/lib/python3.11/site-packages/pretty_midi/pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(


Processed 10966 files so far (latest folder: ../data/raw_data/lmd_full/d)...
Processed 21746 files so far (latest folder: ../data/raw_data/lmd_full/3)...
Processed 32711 files so far (latest folder: ../data/raw_data/lmd_full/f)...
Processed 43522 files so far (latest folder: ../data/raw_data/lmd_full/e)...
Processed 54513 files so far (latest folder: ../data/raw_data/lmd_full/7)...
Processed 65355 files so far (latest folder: ../data/raw_data/lmd_full/a)...
Processed 76204 files so far (latest folder: ../data/raw_data/lmd_full/8)...
Processed 87067 files so far (latest folder: ../data/raw_data/lmd_full/2)...
Processed 98027 files so far (latest folder: ../data/raw_data/lmd_full/b)...
Processed 108907 files so far (latest folder: ../data/raw_data/lmd_full/0)...
Processed 119780 files so far (latest folder: ../data/raw_data/lmd_full/4)...
Processed 130704 files so far (latest folder: ../data/raw_data/lmd_full/c)...
Processed 141553 files so far (latest folder: ../data/raw_data/lmd_full/9

In [4]:
# Save
df.to_csv('../data/processed_data/lmd_full_metadata.csv', index=False)